### Phase 5: XAI Layer: RAG-based Investment Decision Intelligence

In [1]:
# ==================================================
# STEP 0: Setup Paths
# ==================================================

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Since you are inside notebooks folder already
PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"
NEWS_DIR = DATA_DIR / "news_raw"

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

PROJECT_ROOT: C:\Users\jayan\Study Materials\ml_alpha_portfolio\ml-alpha-portfolio-model
NOTEBOOKS_DIR: C:\Users\jayan\Study Materials\ml_alpha_portfolio\ml-alpha-portfolio-model\notebooks


In [3]:
# ==================================================
# STEP 1: Load portfolio decisions
# ==================================================

portfolio_df = pd.read_csv(
    NOTEBOOKS_DIR / "portfolio_decisions.csv",
    parse_dates=["rebalance_date", "holding_start", "holding_end"]
)

print(portfolio_df.head())

  rebalance_date holding_start holding_end      ticker  prediction  weight  \
0     2020-12-03    2020-12-03  2021-01-04    JSWSTEEL    0.019347     0.1   
1     2020-12-03    2020-12-03  2021-01-04  BANKBARODA    0.017761     0.1   
2     2020-12-03    2020-12-03  2021-01-04        ONGC    0.016783     0.1   
3     2020-12-03    2020-12-03  2021-01-04         BEL    0.015401     0.1   
4     2020-12-03    2020-12-03  2021-01-04    ADANIENT    0.014201     0.1   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  


In [4]:
# ==================================================
# STEP 2: Load Features
# ==================================================

def load_feature_data(ticker):
    path = FEATURES_DIR / f"{ticker}.csv"
    
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.rename(columns={"Date": "date"})
    
    return df

In [5]:
# ==================================================
# STEP 3: Load News
# ==================================================

def load_news_data(ticker):
    path = NEWS_DIR / f"{ticker}_news.csv"
    
    df = pd.read_csv(path, parse_dates=["date"])
    return df

In [6]:
# ==================================================
# STEP 4: Filter High-quality sources
# ==================================================

TRUSTED_SOURCES = [
    "reuters", "bloomberg", "cnbc",
    "economictimes", "business-standard",
    "livemint", "moneycontrol",
    "ndtvprofit", "financialexpress",
    "thehindubusinessline"
]

def filter_trusted_news(news_df):
    return news_df[
        news_df["headline"].str.lower().apply(
            lambda x: any(src in x for src in TRUSTED_SOURCES)
        )
    ]

In [7]:
import sys
!{sys.executable} -m pip install sentence-transformers

In [8]:
# ==================================================
# STEP 5: Time-aware News Retrieval
# ==================================================

from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

LOOKBACK_DAYS = 30

def get_relevant_news(ticker, decision_date, query_text, top_k=5):
    
    news_df = load_news_data(ticker)

    news_df = news_df[
        (news_df["date"] <= decision_date) &
        (news_df["date"] >= decision_date - pd.Timedelta(days=LOOKBACK_DAYS))
    ]

    if len(news_df) == 0:
        return []

    news_df = filter_trusted_news(news_df)

    if len(news_df) == 0:
        return []

    # -------- RAG PART --------
    query_embedding = embed_model.encode(query_text)

    news_embeddings = embed_model.encode(news_df["headline"].tolist())

    scores = np.dot(news_embeddings, query_embedding)

    news_df["score"] = scores

    news_df = news_df.sort_values("score", ascending=False)

    return news_df["headline"].head(top_k).tolist()

C:\Users\jayan\anaconda3\envs\ml_alpha_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2053.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# ==================================================
# STEP 6: Feature context
# ==================================================

def get_feature_context(ticker, decision_date):
    
    df = load_feature_data(ticker)
    
    row = df[df["date"] == decision_date]
    
    if len(row) == 0:
        return {}
    
    row = row.iloc[0]
    
    return {
        "momentum_1m": row.get("momentum_1m"),
        "momentum_3m": row.get("momentum_3m"),
        "vol_30d": row.get("vol_30d"),
        "rsi_14": row.get("rsi_14"),
        "macd": row.get("macd"),
        "news_sent": row.get("news_sent"),
        "news_intensity": row.get("news_intensity"),
        "sent_momentum": row.get("sent_momentum"),
        "ret_21d": row.get("ret_21d")
    }

In [10]:
# ==================================================
# STEP 7: Build RAG context
# ==================================================

def build_context(row):
    
    ticker = row["ticker"]
    decision_date = row["rebalance_date"]
    
    feature_data = get_feature_context(ticker, decision_date)

    # Create query for retrieval
    query_text = f"{ticker} stock performance outlook momentum news"

    news = get_relevant_news(ticker, decision_date, query_text)

    context = {
        "ticker": ticker,
        "decision_date": decision_date,
        "holding_period": f"{row['holding_start'].date()} to {row['holding_end'].date()}",
        "rank": row["rank"],
        "prediction": row["prediction"],
        "features": feature_data,
        "news": news
    }
    
    return context

In [11]:
# ==================================================
# STEP 8: Prompt
# ==================================================

def create_batch_prompt(df):

    text = ""
    
    for _, row in df.iterrows():
        context = build_context(row)
        
        text += f"""
Stock: {context['ticker']}
Prediction Score: {context['prediction']}

Key Features:
{context['features']}

Recent News:
{context['news']}

"""
    
    return f"""
You are a professional equity research analyst.

Explain WHY each stock was selected in the portfolio.

STRICT RULES:
- Use both quantitative signals (momentum, RSI, volatility)
- Use news context if relevant
- Be precise (2–3 lines per stock)
- Focus on forward-looking reasoning (next 21 days)

{text}
"""

In [12]:
# !pip install openai

In [13]:
import sys
!{sys.executable} -m pip install openai

In [22]:
#!pip install python-dotenv

In [26]:
import sys
!{sys.executable} -m pip install python-dotenv

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [27]:
# ==================================================
# STEP 9: Generate Explanations
# ==================================================

from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")


def create_batch_prompt(df):

    text = ""
    
    for _, row in df.iterrows():
        context = build_context(row)
        
        text += f"""
Stock: {context['ticker']}
Prediction Score: {context['prediction']}

Key Features:
{context['features']}

Recent News:
{context['news']}

"""
    
    return f"""
You are a professional equity research analyst.

Explain WHY each stock was selected in the portfolio.

STRICT RULES:
- Use both quantitative signals (momentum, RSI, volatility)
- Use news context if relevant
- Be precise (2–3 lines per stock)
- Focus on forward-looking reasoning (next 21 days)

{text}
"""

def generate_explanation(prompt):
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a financial analyst."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content

In [28]:
# ==================================================
# STEP 10: Run ONLY for latest portfolio (1 API call)
# ==================================================

# Get latest rebalance date

latest_date = portfolio_df["rebalance_date"].max()
latest_portfolio = portfolio_df[
    portfolio_df["rebalance_date"] == latest_date
].copy()

latest_portfolio = latest_portfolio.sort_values("rank").head(10)

print("Latest date:", latest_date)
print("Stocks:", len(latest_portfolio))  # should be 10

# Create batch prompt
prompt = create_batch_prompt(latest_portfolio)

# Generate explanation (ONLY ONE CALL)
explanation = generate_explanation(prompt)

print(explanation)

Latest date: 2026-02-02 00:00:00
Stocks: 10
### Stock Analysis and Selection Rationale

**Stock: VBL**  
**Prediction Score: 0.0281**  
VBL shows negative momentum over the past month and three months, with an RSI indicating oversold conditions (36.43). The lack of recent news suggests that the stock may be undervalued, presenting a potential rebound opportunity in the next 21 days as it corrects from its oversold state.

---

**Stock: TRENT**  
**Prediction Score: 0.0241**  
TRENT has experienced significant negative momentum over the last month and three months, compounded by recent news highlighting stalled growth and increased competition. With an RSI of 35.10, the stock is oversold, but the negative sentiment may continue to weigh on it in the near term, making it a risky choice for the next 21 days.

---

**Stock: HINDALCO**  
**Prediction Score: 0.0239**  
HINDALCO has shown positive momentum over the past month and three months, with a healthy RSI of 51.19, indicating a balance

In [25]:
results = [{
    "date": latest_date,
    "explanation": explanation
}]

explain_df = pd.DataFrame(results)
explain_df.to_csv("latest_portfolio_explanations.csv", index=False)